# 03 — Metrics and Losses Demo

- Implementing custom metrics inheriting from `BaseMetric`
- Using `MetricRegistry`
- Implementing custom losses from `TokenLevelLoss` / `SequenceLevelLoss`
- Using `LocalLogger`

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.metrics.base_metrics import BaseMetric, MetricRegistry
from src.losses.base_losses import BaseLoss, TokenLevelLoss, SequenceLevelLoss
from src.logging.local_logger import LocalLogger

## Custom Metric — Exact Match

In [ ]:
class ExactMatch(BaseMetric):
    name = "exact_match"
    def __init__(self):
        super().__init__()
        self._correct = self._total = 0
    def update(self, predictions, references, **kwargs):
        self._correct += sum(p == r for p, r in zip(predictions, references))
        self._total += len(predictions)
    def compute(self):
        return self._correct / self._total if self._total else 0.0
    def reset(self):
        super().reset()
        self._correct = self._total = 0

em = ExactMatch()
em.update(["Paris", "Berlin", "Rome"], ["Paris", "Madrid", "Rome"])
print(f"Exact match: {em.compute():.2f}")  # 2/3

## Custom Token-Level Loss

In [ ]:
class NextTokenLoss(TokenLevelLoss):
    def forward(self, logits, labels, **kwargs):
        return self._cross_entropy(logits[:, :-1].contiguous(), labels[:, 1:].contiguous())

loss_fn = NextTokenLoss(ignore_index=0)
logits = torch.randn(4, 16, 50257)
labels = torch.randint(1, 50257, (4, 16))
print(f"Loss: {loss_fn(logits, labels).item():.4f}")

## LocalLogger Demo

In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as tmpdir:
    lg = LocalLogger(name="demo", log_dir=tmpdir)
    lg.log_config({"lr": 1e-4, "batch_size": 16})
    lg.log_step(0, {"loss": 2.5, "lr": 1e-4})
    lg.log_epoch(0, {"val_loss": 1.4, "exact_match": 0.55})
    lg.close()
    print("Files:", os.listdir(tmpdir))